# Caching Expensive Computations

Green's matrices and stress kernels can take minutes to hours for large faults. GeoDef automatically caches these to disk so that repeated calls with the same inputs load instantly.

This notebook demonstrates:
1. Building a Green's matrix (first call computes, second call loads from cache)
2. Computing a stress kernel (same pattern)
3. Cache management: inspecting, clearing, disabling

In [ ]:
import time

import numpy as np
import geodef

%load_ext autoreload
%autoreload 2

## Setup: Fault and GNSS stations

Same megathrust geometry and station grid as the forward modeling example.

In [ ]:
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=320.0, dip=15.0,
    length=200e3, width=100e3,
    n_length=10, n_width=5,
)

sta_lon, sta_lat = np.meshgrid(
    np.linspace(98.5, 101.5, 8),
    np.linspace(-3.5, -0.5, 8),
)
sta_lat, sta_lon = sta_lat.ravel(), sta_lon.ravel()
n_sta = len(sta_lat)

gnss = geodef.GNSS(
    sta_lat, sta_lon,
    ve=np.zeros(n_sta), vn=np.zeros(n_sta), vu=np.zeros(n_sta),
    se=np.ones(n_sta), sn=np.ones(n_sta), su=np.ones(n_sta),
)
print(f"Fault: {fault.n_patches} patches, GNSS: {gnss.n_stations} stations")

## 1. Green's matrix caching

Start with a clean cache, then call `greens()` twice with the same inputs. The first call computes and saves; the second loads from disk.

In [ ]:
geodef.cache.clear()

# First call: compute from scratch
t0 = time.perf_counter()
G1 = geodef.greens.greens(fault, gnss)
t_compute = time.perf_counter() - t0

# Second call: load from cache
t0 = time.perf_counter()
G2 = geodef.greens.greens(fault, gnss)
t_cached = time.perf_counter() - t0

print(f"Green's matrix shape: {G1.shape}")
print(f"First call (compute):  {t_compute:.4f} s")
print(f"Second call (cached):  {t_cached:.4f} s")
print(f"Speedup:               {t_compute / t_cached:.1f}x")
print(f"Results identical:     {np.array_equal(G1, G2)}")

## 2. Stress kernel caching

Stress kernels evaluate strain Green's functions at every patch centroid (using Okada92/DC3D for internal deformation at depth). This is more expensive than the displacement Green's matrix because it loops over both patches and observation points. Caching is especially valuable here.

In [ ]:
# First call: compute stress kernel
t0 = time.perf_counter()
K1 = fault.stress_kernel()
t_compute = time.perf_counter() - t0

# Second call: load from cache
t0 = time.perf_counter()
K2 = fault.stress_kernel()
t_cached = time.perf_counter() - t0

print(f"Stress kernel shape: {K1.shape}")
print(f"First call (compute):  {t_compute:.4f} s")
print(f"Second call (cached):  {t_cached:.4f} s")
print(f"Speedup:               {t_compute / t_cached:.1f}x")
print(f"Results identical:     {np.array_equal(K1, K2)}")

## 3. Cache management

The cache stores compressed `.npz` files in `.geodef_cache/` (configurable). Each file is named by a SHA-256 hash of the computation inputs, so identical inputs always find the cache and changed inputs always recompute.

In [ ]:
# Inspect the cache
info = geodef.cache.info()
print(f"Cache directory: {geodef.cache.get_dir()}")
print(f"Cached files:    {info['n_files']}")
print(f"Total size:      {info['total_bytes'] / 1024:.1f} KB")

In [ ]:
# Changing any input causes a cache miss and recompute
fault_wider = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=320.0, dip=15.0,
    length=200e3, width=120e3,  # wider fault
    n_length=10, n_width=5,
)

t0 = time.perf_counter()
G_new = geodef.greens.greens(fault_wider, gnss)
t_new = time.perf_counter() - t0

print(f"Different fault -> recomputed in {t_new:.4f} s")
print(f"Cache now has {geodef.cache.info()['n_files']} files")

In [ ]:
# Disable caching (useful for debugging or benchmarking)
geodef.cache.disable()
print(f"Caching enabled: {geodef.cache.is_enabled()}")

t0 = time.perf_counter()
G_uncached = geodef.greens.greens(fault, gnss)
t_uncached = time.perf_counter() - t0
print(f"With caching disabled: {t_uncached:.4f} s (always recomputes)")

# Re-enable
geodef.cache.enable()
print(f"Caching enabled: {geodef.cache.is_enabled()}")

In [ ]:
# Clean up: clear the cache created by this notebook
geodef.cache.clear()
print(f"Cache cleared: {geodef.cache.info()['n_files']} files remaining")